# TabPFN v2 — Telco Churn

TabPFN é um modelo pré-treinado de in-context learning para tabular data.
Não requer treinamento: apenas `fit` (armazena contexto) + `predict`.

**Limitação**: TabPFN v1 aceita ≤ 1000 amostras de treino.
TabPFN v2 (`tabpfn` ≥ 2.0) suporta datasets maiores via batching interno,
mas usaremos subsample estratificado de 2000 amostras por segurança.

In [ ]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from model_utils import load_data, compute_metrics, save_results, print_scores

## 0. Instalar TabPFN

In [ ]:
!pip install tabpfn -q
from tabpfn import TabPFNClassifier
import tabpfn
print(f'tabpfn version: {tabpfn.__version__}')


## 1. Carregar dados

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

Xtr_np  = X_train.values.astype(np.float32)
ytr_np  = y_train.values.astype(int)
Xvl_np  = X_val.values.astype(np.float32)
yvl_np  = y_val.values.astype(int)
Xte_np  = X_test.values.astype(np.float32)
yte_np  = y_test.values.astype(int)

## 2. Subsample estratificado do treino

In [ ]:
# TabPFN funciona melhor com ≤ 2000 amostras de treino
MAX_TRAIN = 2000
if len(Xtr_np) > MAX_TRAIN:
    Xtr_sub, _, ytr_sub, _ = train_test_split(
        Xtr_np, ytr_np,
        train_size=MAX_TRAIN,
        stratify=ytr_np,
        random_state=42
    )
    print(f'Subsample: {Xtr_np.shape[0]} → {Xtr_sub.shape[0]} (estratificado)')
else:
    Xtr_sub, ytr_sub = Xtr_np, ytr_np
    print(f'Usando treino completo: {Xtr_sub.shape[0]} amostras')

print(f'Churn rate subsample: {ytr_sub.mean():.3f}')

## 3. Fit + Predict

In [ ]:
clf = TabPFNClassifier(device='cuda', n_estimators=32)
clf.fit(Xtr_sub, ytr_sub)
print('TabPFN fit concluído')

probs = clf.predict_proba(Xte_np)[:, 1]
preds = (probs >= 0.5).astype(int)
print(f'Predictions geradas para {len(preds)} amostras de teste')

## 4. Avaliar no teste + salvar artefatos

In [ ]:
scores = compute_metrics(yte_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/tabpfn', exist_ok=True)
joblib.dump(clf, 'results/tabpfn/model.pkl')

save_results('tabpfn', yte_np, preds, probs, scores)
print(f'\nNota: modelo treinado com subsample estratificado de {len(Xtr_sub)} amostras.')